# **HACKATHON 1: ASISTENTE RAG CON LLAMA**
## Tema: El tamaño del objeto más grande del mundo

Asistente basado en Llama (vía Ollama, dentro de Colab) que responde preguntas usando RAG sobre una base de conocimiento propia. Compara sus respuestas con y sin RAG.

## **1. CONFIGURACIÓN DEL ENTORNO**

Activa la GPU en **Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU**.

In [1]:
# Verificar que Colab asignó una GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


In [2]:
# Instalar Ollama, arrancar el servidor y descargar Llama 3.1 8B
import subprocess
import time
import requests

!sudo apt-get -qq install -y zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1

servidor = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(30):
    try:
        requests.get("http://localhost:11434")
        print("Servidor de Ollama listo.")
        break
    except requests.exceptions.ConnectionError:
        time.sleep(1)
else:
    print("El servidor no respondió. Vuelve a ejecutar esta celda.")

MODELO = "llama3.1:8b"
!ollama pull {MODELO}

!pip install ollama -q
import ollama
client = ollama.Client(host="http://localhost:11434")
print("Cliente de Ollama inicializado correctamente.")

Servidor de Ollama listo.

Cliente de Ollama inicializado correctamente.


## **2. BASE DE CONOCIMIENTO Y EMBEDDINGS**

Cada fragmento responde a una métrica distinta de 'el más grande'. Verifica las cifras con fuentes primarias antes de presentar.

In [3]:
!pip install sentence-transformers --quiet

from sentence_transformers import SentenceTransformer
import numpy as np

documentos = [
    'El organismo vivo más grande conocido es un hongo del género Armillaria, en el bosque de Malheur (Oregón, Estados Unidos), que se extiende sobre unas 965 hectáreas.',
    'La estructura construida más grande por volumen de concreto es la presa de las Tres Gargantas, en China, de unos 2,3 kilómetros de largo y 185 metros de altura.',
    'El animal más grande que ha existido es la ballena azul, que puede medir hasta unos 30 metros de longitud y pesar más de 150 toneladas.',
    'En astronomía, el objeto más grande conocido es el supercúmulo de Laniakea, una estructura de galaxias que abarca decenas de miles de millones de años luz; no es un objeto único con un límite físico definido.',
    'El tamaño del objeto más grande depende de la métrica: volumen, masa, longitud, área o extensión. Por eso cada respuesta debe indicar qué criterio usa.',
    'La estructura más grande creada por humanos en volumen de concreto es la presa de las Tres Gargantas; no existe un único ganador para todas las métricas de tamaño.'
]

modelo_embeddings = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# normalize_embeddings=True: el producto punto equivale a similitud coseno
embeddings_documentos = modelo_embeddings.encode(documentos, normalize_embeddings=True)
print("Embeddings generados:", embeddings_documentos.shape)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings generados: (6, 384)


## **3. RECUPERACIÓN DE FRAGMENTOS (TOP-K)**

In [4]:
def buscar_fragmentos(pregunta, k=2):
    emb = modelo_embeddings.encode([pregunta], normalize_embeddings=True)
    similitudes = np.dot(embeddings_documentos, emb.T).flatten()
    indices = np.argsort(similitudes)[::-1][:k]
    return [documentos[i] for i in indices]

## **4. ASISTENTE (CON Y SIN RAG)**

Estrategia de prompting: restricción de fuente, instrucción explícita para no responder sin información y límite de longitud.

In [5]:
def asistente(pregunta, usar_rag=True, k=2):
    if usar_rag:
        fragmentos = buscar_fragmentos(pregunta, k=k)
        contexto = "\n".join(f"- {f}" for f in fragmentos)
        prompt = (
            'Responde la pregunta usando SOLO la siguiente información.\n'
            'Si la información no cubre la pregunta, di: "No tengo información suficiente en la base de conocimiento."\n\n'
            f'Información:\n{contexto}\n\n'
            f'Pregunta: {pregunta}\n\n'
            'Respuesta breve (máximo 2 oraciones).'
        )
    else:
        prompt = f'Responde en máximo 2 oraciones: {pregunta}'

    respuesta = client.chat(
        model=MODELO,
        messages=[{"role": "user", "content": prompt}]
    ).message.content
    return respuesta.strip()

## **5. EVALUACIÓN: SIN RAG VS. CON RAG**

La última pregunta es una trampa: no está en la base de conocimiento.

In [6]:
preguntas_prueba = [
    '¿Cuál es el organismo vivo más grande del mundo?',
    '¿Qué presa es la estructura construida más grande por volumen de concreto?',
    '¿Cuál es el animal más grande que ha existido?',
    '¿Cuál es el objeto más grande del universo?',
    '¿Cuál es el edificio más grande de Marte?'
]

resultados = []
for p in preguntas_prueba:
    resultados.append({
        "pregunta": p,
        "sin_rag": asistente(p, usar_rag=False),
        "con_rag": asistente(p, usar_rag=True)
    })

for r in resultados:
    print('PREGUNTA:', r['pregunta'])
    print('  SIN RAG:', r['sin_rag'])
    print('  CON RAG:', r['con_rag'])
    print()

PREGUNTA: ¿Cuál es el organismo vivo más grande del mundo?
  SIN RAG: El organismo vivo más grande del mundo es el árbol gigante, conocido como la "Árbol Gigante de Cristo", ubicado en Queensland (Australia), con más de 130 metros de altura.
  CON RAG: El organismo vivo más grande conocido es un hongo del género Armillaria. Se extiende sobre unas 965 hectáreas en el bosque de Malheur.

PREGUNTA: ¿Qué presa es la estructura construida más grande por volumen de concreto?
  SIN RAG: La presa de Asuán, en Egipto, es la estructura construida más grande por volumen de concreto, con un volumen de más de 38.000.000 de metros cúbicos.
  CON RAG: La estructura construida más grande por volumen de concreto es la presa de las Tres Gargantas, en China. Mide unos 2,3 kilómetros de largo y 185 metros de altura.

PREGUNTA: ¿Cuál es el animal más grande que ha existido?
  SIN RAG: El animal más grande que ha existido es el Argentinosaurus, un dinosaurio herbívoro que vivió en lo que hoy es Argentina du

## **6. CONCLUSIÓN**

Responde en una celda de texto:

1. ¿En qué preguntas la respuesta sin RAG inventó datos o se equivocó de métrica?
2. ¿Qué pasó con la pregunta trampa (Marte) en cada modo?
3. ¿Qué decisión de prompting y de recuperación (k, normalización) tomaste y por qué?
4. ¿Qué limitación tiene tu base de conocimiento (cantidad de fragmentos, verificación de cifras)?

## **7. DEMOSTRACIÓN EN VIVO (CHAT)**

In [7]:
print("Escribe tu pregunta (o 'salir' para terminar).")
while True:
    pregunta_usuario = input("Tú: ").strip()
    if pregunta_usuario.lower() == "salir":
        break
    print("Asistente:", asistente(pregunta_usuario, usar_rag=True))

Escribe tu pregunta (o 'salir' para terminar).
Tú: Porque se les considera los mas grandes sin punto de comparacion clara?
Asistente: No tengo información suficiente en la base de conocimiento.
Tú: Se puede concluir su escala?
Asistente: No tengo suficiente información.
Tú: Salir
